<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

<h1 style="text-align: center;">04 Mobile Algorithms</h1>
<hr>


## **4. Tit-for-Tat Spectrum Management**

<div class="alert alert-light">

  **Name:** (add your name here)
  
  **Matriculation Number:** (add your number here)
  
</div>



### 4.1 Introduction

Distributed systems often require multiple autonomous players to make decisions without central coordination. In mobile computing, devices make local choices about when and how aggressively to use shared wireless spectrum. Each device wants to transmit its data, but its decisions affect interference, waiting times, and the performance of other devices.

We use game theory to study this interaction. Our teaching example is a repeated game between two prisoners, each choosing among **16 levels of informing**. The outcome is a pair of prison sentences, one for each player. Both want fewer years in prison. The familiar two-choice Prisoner's Dilemma appears at the endpoints, but intermediate choices let us explore a much richer set of outcomes.

We compare strategies over 100 rounds, including a generalized Tit-for-Tat strategy, and draw the resulting path in a two-dimensional bargaining domain. This lets us distinguish **Pareto efficiency**, which concerns opportunities to improve outcomes, from **Nash equilibrium**, which concerns each player's incentive to change their own choice. Section 4.3 then explains how to apply this analysis to decisions about shared wireless resources in mobile computing, using communication performance in place of prison sentences.

For background, start with *The Economics of Fair Play* [\[5\]](#ref05), then read Axelrod on repeated cooperation [\[1\]](#ref01). The course collection has an [Algorithms, Game Theory and Control reading guide](https://gitlab.ethz.ch/wireless/BackgroundReading/-/blob/main/Algorithms%20and%20Game%20Theory/README.md).


### 4.2 A Repeated Game with 16 Choices

Game theory helps us study decisions whose outcomes depend on what other participants choose. We keep the prisoner story to make the costs tangible, then connect the same questions about cooperation, incentives, and efficiency to mobile devices sharing a wireless channel.

### 4.2.1 Informing Levels and Prison Sentences

Imagine that the police question two prisoners, A and B, separately. Each chooses an integer informing level from **0 to 15**. Level 0 means staying silent and cooperating fully with the other prisoner; level 15 means fully informing on them. The 14 intermediate levels represent increasing amounts of information disclosed. These levels and the sentence rule below are a constructed teaching model.

An **action** is the level chosen in one round. A **strategy** is the rule used to choose actions over time. Each prisoner has 16 possible actions, giving $16\times16=256$ joint action pairs. A prison sentence is determined by both actions together, rather than chosen directly by either player.

Let $i$ and $j$ be A's and B's levels, and normalize them as $a=i/15$ and $b=j/15$. We assign the following sentences, measured in years:

$$
c_A(i,j)=1-a+4b, \qquad
c_B(i,j)=1+4a-b.
$$

The constant 1 is the sentence under mutual silence. Informing reduces one's own sentence through the term $-a$ or $-b$ and increases the other's sentence through $4a$ or $4b$. This rule gives the following four endpoint outcomes:

| A's level | B's level | A's years in prison | B's years in prison |
|---:|---:|---:|---:|
| 0 | 0 | 1 | 1 |
| 0 | 15 | 5 | 0 |
| 15 | 0 | 0 | 5 |
| 15 | 15 | 4 | 4 |

Intermediate levels produce intermediate sentences. For example, if both choose level 5, then $a=b=1/3$ and each receives $2$ years. Every pair of levels has a specified outcome, and all 256 outcome pairs are distinct in this model.

We continue to call these numbers **payoffs**, but they are costs: **lower is better for both prisoners**. In the usual convention where higher utility is better, we could instead use $u_A=-c_A$ and $u_B=-c_B$. All plots here keep the more concrete unit of years in prison.

<div style="text-align: center; border: 1px solid #9aa3ad; border-radius: 8px; padding: 8px; width: 90%; box-sizing: border-box; margin: 16px auto;">
<img src="../../.admin/notebooks/picture/PrisonnersDilemmaOneYear.png"
     alt="Police question prisoners separately. Sentences: both stay silent, 1 year each; only B informs, A 5 years and B 0; only A informs, A 0 and B 5 years; both inform, 4 years each."
     style="width: 60%; max-width: 1000px; height: auto; filter: grayscale(70%);">
<p><em>The four endpoint outcomes of the 16-level game (levels 0 and 15). A wears red and B wears blue. The numbers are years in prison; a lower sentence is better.</em></p>
</div>

### 4.2.2 Repeating the Game for 100 Rounds

The same two prisoners now face the choice for 100 hypothetical rounds. Round 1 is preset to the one-round Nash equilibrium: both choose level 15 and receive 4 years each. From round 2 onward, both choose simultaneously using their selected strategies and learn the other's level afterward. The sentence rule stays fixed. A strategy can react to past actions, but it cannot see the other player's current choice before making its own.

The rounds model repeated interaction. A sentence shown in a round is that round's outcome, with no running total added to the graph. Repeated play creates an opportunity for cooperation because a choice today can affect the other player's response tomorrow. It does not guarantee cooperation or convergence to an equilibrium.

The illustration below shows the original Tit-for-Tat rule using only levels 0 and 15, starting with cooperation and then copying the other player's previous choice. Our simulator uses a modified version: it starts at the Nash equilibrium and returns toward cooperation gradually, one informing level per round.

<div style="text-align: center; border: 1px solid #9aa3ad; border-radius: 8px; padding: 8px; width: 90%; box-sizing: border-box; margin: 16px auto;">
<img src="../../.admin/notebooks/picture/PrisonnersDilemmaMultiStageOneYear.png"
     alt="Three-round example: A plays Tit-for-Tat. Both stay silent in round 1 and receive 1 year each. In round 2 B informs, giving A 5 years and B 0. In round 3 A responds by informing; B informs again, so both receive 4 years."
     style="width: 80%; max-width: 1200px; height: auto; filter: grayscale(70%);">
<p><em>An endpoint-only example of the 16-level repeated game: A plays Tit-for-Tat, while B informs from round 2 onward. Sentences are shown per round. B's choice in round 2 changes A's response in round 3.</em></p>
</div>

In mobile computing, devices also interact repeatedly through shared resources. Choices such as transmission power or an access attempt rate can have many levels. Cooperation may improve collective performance, while aggressive individual choices can create interference or collisions. The prison-sentence rule is an analogy for conflicting incentives; Section 4.3 explains how to define corresponding actions, outcomes, and adaptation rules for mobile computing.


### 4.2.3 Tit-for-Tat and Cooperation

<div class="alert alert-block alert-info">
<strong>Gradual-cooperation Tit-for-Tat:</strong> Both prisoners start at level 15 in round 1, the one-round Nash equilibrium. From round 2, if the other player's previous level was higher than yours, match it immediately. Otherwise, reduce your own level by one step, stopping at 0. Equal levels therefore invite a small cooperative step: after both choose 15, both try 14 next. This cooperative initiative lets two such players move together toward mutual silence.
</div>

Robert Axelrod explored cooperative strategies in tournaments of the two-choice repeated Prisoner's Dilemma. His 1980 tournament report [\[2\]](#ref02), the 1981 paper with W. D. Hamilton [\[3\]](#ref03), and his 1984 book *The Evolution of Cooperation* [\[1\]](#ref01) describe this work. Tit-for-Tat won his original tournaments, although its success depends on the opponents and game conditions.

The original Tit-for-Tat strategy cooperates first and then copies the opponent's previous action. Here we use a modified 16-level rule with a preset Nash starting point, gradual cooperation, and cooperative initiative when both players used the same level. With $i_t$ and $j_t$ denoting the levels in round $t$, A's rule is

$$
i_1=15, \qquad i_t=\begin{cases}
j_{t-1}, & j_{t-1}>i_{t-1},\\
\max(0,i_{t-1}-1), & j_{t-1}\le i_{t-1},
\end{cases}\quad\text{for }t\ge2.
$$

B uses the corresponding rule if B also selects Tit-for-Tat. Both players respond to the previous round before either new choice is revealed. Two Tit-for-Tat players choose levels 15, 14, 13, and so on, reaching mutual silence at level 0 in round 16 and staying there. Their equal sentences descend from $(4,4)$ to $(1,1)$ in steps of 0.2 years per player, along the Equal Outcomes diagonal. Against Always cooperate, A also reaches level 0 in round 16. Against Always defect, A alternates between a cooperative probe at level 14 and a response at level 15, rather than continuing toward silence while the other player keeps informing fully.

The original strategy is **nice**, because it starts cooperatively; **retaliatory**, because it responds in the following round; **forgiving**, because it returns to cooperation after the other player does; and **clear**, because its copying rule is easy to understand. Our modified rule retains an immediate response to greater informing, adds a one-step cooperative initiative at equal levels, and returns toward cooperation gradually. Its preset first-round defection differs from the original cooperative start, and the historical tournament results do not establish that this modified rule is optimal.

Mobile computing provides a practical scenario for distributed systems: devices repeatedly share wireless channels and make local decisions that affect one another. Protocols can encourage cooperation, such as backing off when a channel is busy. A wireless application of social-action concepts appears in the spectrum-sharing paper [\[6\]](#ref06). Tit-for-Tat copies an observed action, so noise or a mistaken observation can also trigger unnecessary retaliation.


### 4.2.4 The Bargaining Domain and Pareto Efficiency

So far, we have described outcomes as pairs of prison sentences. We can also draw each pair as a point: **A's years in prison on the horizontal axis and B's on the vertical axis**. A movement left improves A's outcome; a movement down improves B's. Both would prefer an attainable point toward the lower left.

The **feasible outcome set**, which we use here as the **bargaining domain**, contains every pair that the sentence rule can produce:

$$
\mathcal{F}=\{(c_A(i,j),c_B(i,j)):i,j\in\{0,1,\ldots,15\}\}.
$$

There are 256 points. The axes show outcomes, rather than informing levels. Connecting two successive outcomes traces the order of play; points along a connecting line need not themselves be attainable in one round. If we allowed lotteries over joint actions or averages over many rounds, convex combinations would form a larger feasible set. The graph marks the actual single-round outcomes.

The word bargaining highlights the possibility of comparing outcomes that both players might agree to pursue. A full bargaining problem also specifies what happens if no agreement is reached. We have defined no negotiation protocol or binding agreement here, so the simulation follows the chosen strategies rather than computing a bargaining solution. See the introduction to bargaining in [\[11\]](#ref11).

An outcome **Pareto dominates** another if it gives neither prisoner more years and gives at least one strictly fewer years. For example, $(1,1)$ dominates $(4,4)$: both sentences fall by three years. An outcome is **Pareto efficient** if no other feasible outcome dominates it. Equivalently, there is no feasible change that helps at least one player without hurting the other. This is the usual Pareto criterion, with the inequalities reversed because our payoffs are costs [\[11\]](#ref11).

In this model, enumeration of all 256 outcomes gives **31 Pareto-efficient points**: those obtained when at least one prisoner chooses level 0. They trace the lower-left frontier from $(0,5)$ through $(1,1)$ to $(5,0)$. The graph highlights these points in green. The lines between them are visual guides to the frontier.

**Efficiency does not imply fairness.** The extreme outcome $(0,5)$ is Pareto efficient: A cannot improve on zero years, and reducing B's sentence requires increasing A's. It is nevertheless very unequal. The symmetric outcome $(1,1)$ is also efficient. The Pareto criterion identifies remaining opportunities for improvement; choosing among efficient outcomes requires an additional objective or agreement.

The gray dashed diagonal, labeled **Equal Outcomes**, shows $c_A=c_B$: both prisoners receive the same sentence. It is a 45-degree line because both axes use the same scale. Our game has 16 attainable equal outcomes between $(1,1)$ and $(4,4)$; the rest of the line is a reference guide. Equality is one possible fairness criterion and does not guarantee efficiency: $(4,4)$ is equal but inefficient, whereas $(1,1)$ is the only equal outcome on the Pareto frontier.

### 4.2.5 Nash Equilibrium: Can One Player Improve Alone?

Pareto efficiency compares feasible outcomes for both players. **Nash equilibrium** asks a different question: with the other player's action fixed, can either player reduce their own sentence by changing their own action alone? A pair is a Nash equilibrium of the one-round game when the answer is no for both players. Each action must be a **best response** to the other. Nash's original paper [\[9\]](#ref09) and the [MIT lecture notes on Nash equilibrium](https://ocw.mit.edu/courses/14-12-economic-applications-of-game-theory-fall-2025/mit14_12f12_chapter6_f12.pdf) develop this idea.

For costs, an equilibrium pair $(i^*,j^*)$ satisfies

$$
c_A(i^*,j^*)\le c_A(i,j^*)\quad\text{for every }i,
\qquad
c_B(i^*,j^*)\le c_B(i^*,j)\quad\text{for every }j.
$$

In our sentence rule, increasing A's level by one changes A's own sentence by $-1/15$, which is negative regardless of B's level. Thus A's best response to any fixed level of B is level 15. The same reasoning applies to B. The **unique one-round Nash equilibrium** is therefore **$(i,j)=(15,15)$**, producing **$(c_A,c_B)=(4,4)$**. The graph marks this outcome with a red diamond. The code checks every possible unilateral change to identify equilibria.

This equilibrium is **not Pareto efficient**: both would prefer the feasible outcome $(1,1)$. Yet mutual silence is not a one-round equilibrium. If B stays at level 0, A can shorten A's sentence by informing more, while B's sentence increases. Improving together and having no incentive to change alone are distinct properties.

The marked diamond concerns the **one-round game**. A strategy in the repeated game specifies actions after every possible history, so evaluating a repeated-game Nash equilibrium requires considering future responses and total outcomes. A 100-round trajectory does not itself prove equilibrium or convergence. With a known final round, complete information, common knowledge that both players rationally minimize total sentences, and no binding agreements, backward induction predicts level 15 in every round: there is no future retaliation in the last round, and the same reasoning works backward. Our simulation compares specified behavioral strategies rather than solving that repeated-game equilibrium.

The **Nash bargaining solution** is a separate concept that selects an agreement using a feasible set and a disagreement outcome [\[11\]](#ref11). The red diamond identifies the one-round Nash equilibrium, rather than a negotiated compromise.

### 4.2.6 Exploring a 100-Round Path

Every strategy starts with the same preset round 1: both prisoners choose level 15, giving the Nash outcome $(4,4)$. From round 2, **Always cooperate** chooses level 0, **Always defect** chooses level 15, **Fixed level** uses the corresponding level slider, and **Random** chooses uniformly among all 16 levels. **Tit-for-Tat** uses the modified rule in Section 4.2.3: match an opponent's previous level immediately if it was higher than your own; otherwise lower your own level by one step, stopping at 0. Equal levels also trigger this cooperative step. The dropdown retains the short name Tit-for-Tat for this variant.

The graph shows all 256 outcomes, the Pareto frontier, the one-round Nash equilibrium, and the path generated by 100 rounds. The trajectory uses blue curved arrows between successive outcomes. Small variations in the bend make repeated jumps easier to distinguish, and opposite-direction jumps curve on opposite sides. The curves are visual connectors: outcomes remain at their exact coordinates, and the drawing variation does not change the game. An open circle marks the first outcome and an X marks the last displayed outcome. There are **100 round outcomes and 99 transitions**; repeated outcomes overlap, so a stationary path may appear as a single point.

By default, both prisoners use **Random** and the graph shows the first **10 rounds**. Round 1 is preset to the one-round Nash equilibrium for every strategy pairing, including Tit-for-Tat. The legend labels this reference outcome **1-round NE**. From round 2, the selected strategies determine whether the path leaves the equilibrium or stays there.

Use **Show through round** to reveal the existing path one round at a time, up to all 100 rounds. This does not generate new random choices. A seed keeps random runs reproducible; **Run** changes the seed. Changing a strategy or level recalculates all 100 rounds using the displayed seed. Compare Random against Random with Tit-for-Tat, mutual cooperation, mutual defection, and fixed intermediate levels. Reaching the green frontier depends on the selected strategies; it is not guaranteed.


In [ ]:
from pathlib import Path

_helper = next(root / ".admin/notebooks/tit_for_tat.py"
               for root in (Path.cwd(), *Path.cwd().parents)
               if (root / ".admin/notebooks/tit_for_tat.py").is_file())
get_ipython().run_line_magic("run", f'-i "{_helper.as_posix()}"')


### 4.3 Applying the Multistage Game to Mobile Computing

The previous analysis gives us a way to study mobile devices that repeatedly make local decisions while sharing a wireless resource. Imagine two nearby mobile devices, each transmitting to its own receiver. Each device wants a reliable connection, but its transmissions also affect the other link. We could model each device as choosing among 16 transmission-power settings at regular intervals. Increasing power can strengthen its own received signal while increasing interference at the other receiver. Cooperation would mean using a setting that meets its communication needs while limiting interference to the other link.

To apply the game, we must first replace the prison sentences with relevant communication outcomes. For example, we could measure each device's average packet delay during an interval and use the pair of delays as the outcome. Both devices would prefer smaller values, just as both prisoners preferred fewer years. We would obtain these outcomes from measurements or a wireless model for each pair of settings. The prison-sentence formula does not describe radio performance. If we chose throughput instead, larger values would be better and the favorable direction in the graph would reverse.

The two-dimensional bargaining domain would then show the communication outcomes the devices can achieve together. Its Pareto frontier would identify settings where one device's delay cannot be reduced without increasing the other's. The Equal Outcomes diagonal would mean equal delays. For two devices with similar needs, this might be a useful fairness target; devices carrying different traffic or serving different deadlines may require a different allocation. A one-round Nash equilibrium would be a pair of settings where neither device could reduce its own delay by changing its setting alone. Depending on the actual wireless model, such an equilibrium could be less efficient than a coordinated choice. Its location must be calculated from the communication outcomes, rather than inherited from the prisoner game.

Repeated interaction adds the trajectory we explored in Section 4.2. Instead of choosing a setting once, each device can observe what happened and adjust its next choice. A cooperative policy could try a small reduction in transmission power and keep it if communication remains satisfactory and the shared outcome improves. When both devices participate, a sequence of small adjustments could move them toward a better operating point. Our modified Tit-for-Tat rule illustrates this idea of taking a cooperative step and responding to the other player's behavior. It suggests a possible design principle; it does not establish that copying the other device's power or increasing power in response will improve a real network.

A practical policy also needs a way to observe the other device's behavior, such as exchanged information or agreed feedback. Poor reception alone does not reveal selfish behavior: movement, obstacles, fading, and changing traffic can also alter the outcome. As devices move, the feasible outcome set itself can change, so the same settings may no longer produce the same point. This makes measurement, gradual adaptation, and recovery from mistaken reactions important parts of distributed coordination.

Wi-Fi provides a familiar example of coordination through local observations: devices sense the channel and use random backoff to manage access. These mechanisms respond to channel conditions and transmission outcomes rather than copying a particular opponent's previous action. Our **Jemula802** simulator provides a way to study these interactions. [Notebook 03.7: Wi-Fi Simulation with Jemula802](../03_Mobile_Systems/03.7_Wi-Fi_Simulation_with_Jemula802.ipynb) introduces its configurable scenarios and the analysis of throughput and packet delay. By comparing different joint configurations under the same traffic and radio conditions, we can obtain the communication outcomes needed to construct a bargaining domain and assess efficiency and fairness. An adaptive strategy would then need to be implemented and evaluated in Jemula802. The spectrum-sharing paper [\[6\]](#ref06) provides further application context.

The useful connection to mobile computing is the method of analysis: define the devices' choices and objectives, determine which joint outcomes are feasible, distinguish efficient outcomes from individually stable ones, and examine how repeated decisions move the system through that domain. A successful coordination rule should be assessed for communication performance, fairness, and its ability to adapt when conditions change.

#### 4.3.1 Regulator and Device Objectives: Efficiency, Overhead, and QoS

The **radio regulator** considers the use of spectrum across many systems and services. Its goals include enabling efficient use of the available spectrum, protecting authorised services from harmful interference, and supporting coexistence. A device improving its own connection at the expense of other services may therefore conflict with the wider objective. Regulation defines the boundaries within which devices can act, such as permitted spectrum access and transmission limits. In the cognitive-radio setting of the spectrum-sharing paper [\[6\]](#ref06), two central concerns are protecting incumbent services and allowing the other radio systems to coexist.

An **individual device** acts on behalf of its application. It may need sufficient throughput for a video stream, low delay and delay variation for an interactive call, or reliable delivery for a sensor update. These requirements form its **quality of service (QoS)** goals. The device also has an interest in limiting its energy use. Meeting a throughput requirement is different from always seeking the highest possible throughput: once the application's demand is satisfied, additional access may bring little benefit while reducing the opportunities available to other devices. A device that considers only its own performance has little reason to account for that external cost.

A **network operator or coordination protocol** must turn these individual requirements into an allocation of resources. It manages access, scheduling, and feedback so that devices can meet their needs while the network operates within regulatory constraints. This is where detailed QoS and throughput decisions are made; spectrum regulation itself does not schedule each packet or guarantee every application's QoS.

**Spectrum efficiency** concerns the useful communication achieved with a limited frequency resource. For a fixed bandwidth $B$ and observation interval, one useful system-level measure is the total successfully delivered payload rate divided by bandwidth:

$$
\eta_{\mathrm{payload}}=\frac{\sum_i G_i}{B}\qquad\text{in bit/s/Hz},
$$

where $G_i$ is device $i$'s **data throughput**, meaning newly delivered application payload bits per second. This distinguishes useful throughput from the physical transmission rate: headers and control traffic are excluded, and payload is counted once when it is successfully delivered, even if several transmission attempts were needed. A high value of the aggregate measure also says nothing about how that data throughput is distributed or whether each device meets its delay and reliability requirements.

**Overhead** is the resource cost of making communication work. Packet headers, acknowledgements, coordination messages, and scheduling information occupy airtime. Sensing and backoff also consume time or energy; collisions and repeated transmission attempts consume extra resources per successfully delivered packet. Some coordination overhead is worthwhile: it can prevent collisions, protect other services, or make delay more predictable. Reducing overhead blindly can therefore reduce useful throughput or make QoS worse, while excessive coordination can leave too few resources for payload. The objective is an effective balance, rather than either maximum channel activity or minimum overhead alone.

These tensions appear explicitly in the model in the spectrum-sharing paper [\[6\]](#ref06). Its evaluation shows that total spectrum utilisation can be maximised when one radio system monopolises access, even though the others are poorly served. It also examines a fixed lower bound on contention-window size: this can prevent monopolisation, but a bound that is poorly matched to the scenario leaves spectrum underused. The example separates a regulatory constraint, an individual application's requirement, and the efficiency of the resulting shared operation.

In terms of our multistage game, regulatory rules constrain the available actions, device QoS requirements shape what each player considers a desirable outcome, and protocol overhead affects which outcomes are achievable. Cooperative adaptation should seek an efficient allocation that satisfies application requirements and protects other users. Equality alone is insufficient when applications need different resources, and an individually stable Nash equilibrium need not satisfy all QoS goals. With **Jemula802**, we can begin by comparing throughput and delay under different joint configurations, then include measurements of airtime costs, energy, and unmet QoS requirements when evaluating a proposed coordination rule.

#### 4.3.2 Licensed Spectrum, Unlicensed Spectrum, and Open Access

In **licensed spectrum**, an operator or other authorised user receives defined rights to use frequencies under specified conditions. These can concern the band, location, transmission power, and protection of other services. Access rights may be exclusive or shared, depending on the licensing arrangement. A mobile phone normally operates under its network operator's authorisation; each subscriber does not obtain a separate spectrum licence. Defined access rights help an operator plan coverage, capacity, and QoS, but do not by themselves guarantee a particular data throughput or delay for every device.

In **unlicensed**, or **licence-exempt**, spectrum, compliant devices can operate without an individual spectrum licence, subject to the rules for that band. Many Wi-Fi and Bluetooth uses follow this approach. Unlicensed does not mean unregulated: power limits, equipment requirements, and sharing conditions still apply. Devices must coexist with other permitted users, so an application cannot assume exclusive airtime or freedom from interference. This makes contention, adaptation, and cooperation especially important when independent systems share the same band.

In this discussion, **open access** means that spectrum access is available to devices meeting the common conditions, rather than reserved through an individual assignment. It concerns access to the radio resource, not whether a particular Wi-Fi network allows anyone to join. It also does not mean that every device may transmit whenever it wants or at any power. An open sharing arrangement still needs rules that limit harmful interactions and discourage monopolisation.

Licensed and unlicensed access are not the only possible arrangements. **Shared licensed access** can give several authorised systems access under defined conditions. In the cognitive-radio scenario discussed in the spectrum-sharing paper [\[6\]](#ref06), additional radios seek opportunities in spectrum used by incumbent services while protecting those services and coexisting with one another. This opportunistic use requires an appropriate authorisation framework; detecting an idle frequency alone does not create permission to use it. Incumbent protection adds a constraint to the secondary radios' game, and may require them to reduce or stop transmissions when the protected service needs the resource.

The connection to our multistage analysis is that **the access regime defines who can play and which actions are permitted**. Within those boundaries, devices still seek QoS and data throughput, while the regulator seeks efficient use and coexistence. Licensed access can support planned coordination, whereas open shared access places more of the coordination burden on the interacting systems. In either case, coordination consumes resources but can reduce interference and contention. The useful comparison is therefore whether the resulting data throughput, QoS, fairness, and overhead justify the sharing arrangement, rather than assuming that either licensed or unlicensed access is always more efficient.



### 4.4 Concluding Remarks

The 16-level repeated game shows how local decisions and responses to past behavior create a trajectory through a two-player outcome space. Pareto efficiency asks whether an outcome can be improved without making someone worse off. Nash equilibrium asks whether either player can improve by changing their own action alone. Equal outcomes describe how costs are shared. These are different properties, and none alone establishes that a distributed system performs well.

For mobile computing, the same analysis begins with communication choices and measurable outcomes. Repeated cooperation may help devices share wireless resources, but the outcome depends on the physical channel, traffic, available observations, and adaptation rule. Our gradual-cooperation strategy demonstrates one possible behavior in the teaching game; its performance in a wireless system would require a suitable model and evaluation.

In the preceding PID notebooks, a controller uses measurements to adjust an input. Here, several decision makers affect one another through a shared resource, so an adjustment that helps one may hurt another. A learning controller would therefore need objectives that account for both individual performance and cooperation, together with adaptation to changing conditions. Notebook 05.10 builds on repeated interaction by introducing an AI advisor.


### 4.5 Further Readings

The course [Algorithms, Game Theory and Control collection](https://gitlab.ethz.ch/wireless/BackgroundReading/-/blob/main/Algorithms%20and%20Game%20Theory/README.md) contains the saved readings linked below. Start with [\[5\]](#ref05) for an accessible introduction, [\[1\]](#ref01) for reciprocity, and [\[6\]](#ref06) for the wireless application.

[<a id="ref01">1</a>] R. Axelrod (1984) [**The Evolution of Cooperation.**](https://gitlab.ethz.ch/wireless/BackgroundReading/-/blob/main/Algorithms%20and%20Game%20Theory/1984-evolution-of-cooperation.pdf) Basic Books.

[<a id="ref02">2</a>] R. Axelrod (1980) [**Effective Choice in the Prisoner's Dilemma.**](https://doi.org/10.1177/002200277902400101) *Journal of Conflict Resolution*, 24(1), 3–25. 

[<a id="ref03">3</a>] R. Axelrod and W. D. Hamilton (1981) [**The Evolution of Cooperation.**](https://doi.org/10.1126/science.7466396) *Science*, 211(4489), 1390–1396.


[<a id="ref04">4</a>] Veritasium (2020) [**This Game Theory Problem Will Change the Way You See the World.**](https://www.youtube.com/watch?v=mScpHTIi-kM) *YouTube*, uploaded by Veritasium, 14 Mar. 2020.

[<a id="ref05">5</a>] Sigmund, Fehr and Nowak (2002). [**The Economics of Fair Play.**](https://gitlab.ethz.ch/wireless/BackgroundReading/-/blob/main/Algorithms%20and%20Game%20Theory/2002-economics-of-fair-play.pdf) An accessible introduction to fairness, social preferences and cooperation in economic games.

[<a id="ref06">6</a>] S. Mangold, L. Berlemann and S. Nandagopalan (2006). [**Spectrum Sharing with Value-Orientation for Cognitive Radio.**](https://gitlab.ethz.ch/wireless/BackgroundReading/-/blob/main/Algorithms%20and%20Game%20Theory/2006-spectrum-sharing-value-orientation.pdf) Connects independent radio decisions, social-action concepts, incumbent protection and coexistence.

[<a id="ref07">7</a>] (2012). [**Why We Help.**](https://gitlab.ethz.ch/wireless/BackgroundReading/-/blob/main/Algorithms%20and%20Game%20Theory/2012-why-we-help.pdf) *Scientific American*, July. Further reading on cooperation and helping behavior in evolution.

[<a id="ref08">8</a>] (2002). [**The Good, the Bad and the Lonely.**](https://gitlab.ethz.ch/wireless/BackgroundReading/-/blob/main/Algorithms%20and%20Game%20Theory/2002-good-bad-and-lonely.pdf) *Nature* commentary. Extends the discussion to voluntary participation in public-goods games, where players may cooperate, defect or opt out.

[<a id="ref09">9</a>] John Nash (1951). [**Non-Cooperative Games.**](https://gitlab.ethz.ch/wireless/BackgroundReading/-/blob/main/Algorithms%20and%20Game%20Theory/1951-nash-non-cooperative-games.pdf) Original mathematical treatment of equilibrium in non-cooperative games; background for Section 4.2.5.

[<a id="ref10">10</a>] [**Evolutionary Game Strategies.**](https://gitlab.ethz.ch/wireless/BackgroundReading/-/blob/main/Algorithms%20and%20Game%20Theory/undated-evolutionary-game-strategies.ppt) Optional lecture slides on the Prisoner’s Dilemma, deterministic strategies and tournaments; publication year not verified.

[<a id="ref11">11</a>] A. Goel (2020). [**Introduction to game-theoretic concepts, Lectures 2 and 3.**](https://web.stanford.edu/~ashishg/msande336/aut2021/win2020/msande336_handouts/lecture2-3.pdf) Stanford MS&E 336/CS 366, notes by Raymond Ye Lee. Introduces Nash equilibrium, feasible bargaining sets, Pareto efficiency, and the separate Nash bargaining solution.


----
### 4.6 Hands-On Exercises

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>

The first two questions are required. All remaining questions are optional.


**If B stays silent (level 0), which level gives A the shortest prison sentence?**

Answer:


**Why do both players fully inform in the one-round Nash equilibrium, even though mutual silence would give both shorter sentences?**

Answer:


**Set both players to Tit-for-Tat. How do their prison sentences change over time?**

Answer:


**OPTIONAL: Set A to Tit-for-Tat and B to Random. Describe how A reacts to B.**

Answer:


**OPTIONAL: How could two mobile devices cooperate to meet their QoS needs while sharing spectrum?**

Answer:


**OPTIONAL: What could be improved in this notebook?**

Answer:
